In [2]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn as nn
#import torch.optim as optim

In [8]:
class LSTM(nn.Module):
  def __init__(self, input_size, hidden_size, device):
    super(LSTM, self).__init__()
    self.device = device
    self.params = nn.ParameterList(self.init_params(input_size, hidden_size))
    """
    Inputs:
      input_size: int, feature dimension of input sequence
      hidden_size: int, feature dimension of hidden state
      device: torch.device()
    """

  def init_params(self, input_size, hidden_size):
    """
    Inputs:
      input_size: int, feature dimension of input sequence
      hidden_size: int, feature dimension of hidden state

    Outputs:
      Weights for proposal: W_xc, W_hc, b_c
      Weights for input gate: W_xi, W_hi, b_i
      Weights for forget gate: W_xf, W_hf, b_f
      Weights for output gate: W_xo, W_ho, b_o
    """

    W_xc, W_hc, b_c = torch.randn(input_size, hidden_size) * 0.1, torch.randn(hidden_size, hidden_size) * 0.1, torch.zeros(hidden_size)
    W_xi, W_hi, b_i = torch.randn(input_size, hidden_size) * 0.1, torch.randn(hidden_size, hidden_size) * 0.1, torch.zeros(hidden_size)
    W_xf, W_hf, b_f = torch.randn(input_size, hidden_size) * 0.1, torch.randn(hidden_size, hidden_size) * 0.1, torch.zeros(hidden_size)
    W_xo, W_ho, b_o = torch.randn(input_size, hidden_size) * 0.1, torch.randn(hidden_size, hidden_size) * 0.1, torch.zeros(hidden_size)

    params = [W_xc, W_hc, b_c, W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o]
    return params


  def lstm(self, X, state):
    """
    Inputs:
      X: tuple of tensors (src, src_len). src, size (N, D_in) or (N, T, D_in), where N is the batch size,
        T is the length of the sequence(s). src_len, size of (N,), is the valid length for each sequence.

      state: tuple of tensors (h, c). h, size of (N, hidden_size) is the hidden state of LSTM. c, size of
            (N, hidden_size), is the memory cell of the LSTM.

    Outputs:
      o: tensor of size (N, T, hidden_size). Contains the output features (the hidden state H_t) for each t.
      state: the same as input state. Contains the hidden state H_T and cell state C_T for the last timestep T.
    """

    src, src_len = X
    h, c = state

    # make sure always has a T dim
    if len(src.shape) == 2:
      src = src.unsqueeze(1)

    N, T, D_in = src.shape
    W_xc, W_hc, b_c, W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o = self.params
    o = []

    #have to iterate through the sequences in parallel word by word (T)
    for t in range(T):
      x_t = src[:, t, :] #get the t^{th} input

      i_t = torch.sigmoid(x_t @ W_xi + h @ W_hi + b_i) #input gate
      f_t = torch.sigmoid(x_t @ W_xf + h @ W_hf + b_f) #forget gate
      o_t = torch.sigmoid(x_t @ W_xo + h @ W_ho + b_o) #output gate
      c_t = torch.tanh(x_t @ W_xc + h @ W_hc + b_c) #proposal

      c = f_t * c + i_t * c_t #state update
      h = o_t * torch.tanh(c_t) #hidden update

      o.append(h.unsqueeze(1))
    o = torch.cat(o, dim=1)
    state = (h, c)
    return o, state

  def forward(self, inputs, state):
    return self.lstm(inputs, state)

In [4]:
test_lstm = LSTM(10, 5, torch.device('cpu'))
test_src = torch.ones(12, 8, 10)
test_src_len = torch.ones(12) * 8
test_h = torch.zeros(12, 5).float()
test_c = torch.zeros(12, 5).float()

test_o, test_state = test_lstm((test_src, test_src_len), (test_h, test_c))

print(test_o.shape)
print(test_state[0].shape)
print(test_state[1].shape)

torch.Size([12, 8, 5])
torch.Size([12, 5])
torch.Size([12, 5])


In [5]:
def masked_softmax(X, valid_length):
  """
  inputs:
    X: 3-D tensor
    valid_length: 1-D or 2-D tensor
  """
  mask_value = -1e7

  if len(X.shape) == 2:
    X = X.unsqueeze(1)

  N, n, m = X.shape

  if len(valid_length.shape) == 1:
    valid_length = valid_length.repeat_interleave(n, dim=0)
  else:
    valid_length = valid_length.reshape((-1,))

  mask = torch.arange(m)[None, :].to(X.device) >= valid_length[:, None]
  X.view(-1, m)[mask] = mask_value

  Y = torch.softmax(X, dim=-1)


  return Y

In [9]:
class DotProductAttention(nn.Module):
  def __init__(self):
      super(DotProductAttention, self).__init__()

  def forward(self, query, key, value, valid_length=None):
    """
    inputs:
      query: tensor of size (B, n, d)
      key: tensor of size (B, m, d)
      value: tensor of size (B, m, dim_v)
      valid_length: (B, )

      B is the batch_size, n is the number of queries, m is the number of <key, value> pairs,
      d is the feature dimension of the query, and dim_v is the feature dimension of the value.

    Outputs:
      attention: tensor of size (B, n, dim_v), weighted sum of values
    """
    temp = torch.bmm(query, key.transpose(1,2)) / torch.sqrt(torch.tensor(query.size(-1)))
    temp_soft = masked_softmax(temp, valid_length)
    attention = torch.bmm(temp_soft, value)

    return attention

In [10]:
class MLPAttention(nn.Module):
  def __init__(self, d_v, d_k, d_q):
    super(MLPAttention, self).__init__()
    """
    Inputs:
      d_k: feature dimension of key
      d_v: feature dimension of vector v
      d_q: feature dimension of query
    """

    self.W_k = nn.Linear(d_k, d_v)
    self.W_q = nn.Linear(d_q, d_v)
    self.v = nn.Linear(d_v, 1)

  def forward(self, query, key, value, valid_length):
    """
    inputs:
      query: tensor of size (B, n, d)
      key: tensor of size (B, m, d)
      value: tensor of size (B, m, dim_v)
      valid_length: either (B, )

      B is the batch_size, n is the number of queries, m is the number of <key, value> pairs,
      d is the feature dimension of the query, and dim_v is the feature dimension of the value.

    Outputs:
      attention: tensor of size (B, n, dim_v), weighted sum of values
    """

    q_proj = self.W_q(query)
    k_proj = self.W_k(key)
    weights = torch.tanh(q_proj.unsqueeze(2) + k_proj.unsqueeze(1))
    Y = self.v(weights)
    Y = masked_softmax(Y.squeeze(3), valid_length)
    Y = torch.bmm(Y, values)

    return Y

In [ ]:
years = sorted(df['year'].unique())
test_year = years[-1]            # final test
val_years = years[:-1]           # for LOYO validation

def make_windows(df_years, L, H, scaler=None, fit_scaler=False, allow_cross_year_input=True, target_col='y'):
    # df has columns: ['series_id','year','timestamp', features..., 'y'] and is sorted by series_id, timestamp
    # returns X: (N, L, F), y: (N,) or (N, H)
    # implement causal rolling and prevent targets from using future info
    ...

best_hp = None
best_score = float('inf')

# ----- LOYO validation (model selection) -----
for v in val_years:
    train_years = [y for y in val_years if y != v]  # exclude validation year
    Xtr, ytr, scaler = make_windows(df[df.year.isin(train_years)], L, H, scaler=None, fit_scaler=True)
    Xval, yval, _      = make_windows(df[df.year==v],            L, H, scaler=scaler, fit_scaler=False)

    model = LSTMModel(**hp)        # try a grid/random set of hp
    best_fold_val = train_with_early_stopping(model, Xtr, ytr, Xval, yval)  # returns best val metric
    log(v, hp, best_fold_val)

# pick hyperparams with the best average across v
best_hp = select_hp_from_logs()

# ----- Final train & test -----
train_years = [y for y in years if y != test_year]
Xtr, ytr, scaler = make_windows(df[df.year.isin(train_years)], L, H, scaler=None, fit_scaler=True)
Xte, yte, _      = make_windows(df[df.year==test_year],        L, H, scaler=scaler, fit_scaler=False)

final_model = LSTMModel(**best_hp)
train_with_early_stopping(final_model, Xtr, ytr, Xval=None, yval=None)  # or split a tiny train/val inside training years
test_metrics = evaluate(final_model, Xte, yte)
print(test_metrics)